# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook demonstrates step-by-step how to load and explore the FAIR² colorectal cancer survivors dataset with the [`mlcroissant`](https://github.com/mlcommons/croissant-python) library, using schema-defined entity IDs throughout.

### Dataset Source
The dataset is provided via a Croissant schema URL and contains tabular records on clinicopathological and molecular features of second primary colorectal cancer in survivors. This notebook shows how to:
- Load the metadata and schema
- Inspect available record sets and fields
- Extract data by record set `@id`
- Perform basic EDA and visualization using only entity IDs for columns

In [ ]:
# Ensure `mlcroissant` is installed
!pip install --quiet mlcroissant

## 1. Data Loading

We load the dataset schema metadata and display a short description.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the Croissant dataset via its schema URL
dataset = mlc.Dataset(croissant_url)

# Print dataset summary from its metadata object
meta = dataset.metadata
print(f"{meta.name}: {meta.description}")


## 2. Data Overview
Let's list the available record sets, their `@id`, and their field `@id`s. We'll use these IDs for all references, as required by the Croissant convention.

In [ ]:
# List all record sets and their fields by `@id`
# This assumes standard Croissant recordSet and field structure
record_sets = dataset.metadata.record_sets

if not record_sets:
    print('No record sets found in the metadata. Please check schema or Croissant version.')
else:
    for rs in record_sets:
        print(f'Found record set: @id = {rs.id}, name = {getattr(rs, "name", None)}')
        if rs.fields:
            print(' Available fields:')
            for field in rs.fields:
                print(f'  - @id = {field.id}, name = {getattr(field, "name", None)}, dataType = {getattr(field, "data_type", None)}')
        else:
            print(' No fields listed for this record set.')

### List a few records from the main record set

We'll print several rows from the primary record set. Make sure to use its `@id` for all future operations.

In [ ]:
# Identify the main record set @id (by examining output above)
# Here, we programmatically select the first record set for demonstration
if not record_sets:
    print('No record sets to display records for.')
else:
    main_rs = record_sets[0]  # Replace this index as needed
    main_record_set_id = main_rs.id
    print(f'First record set @id: {main_record_set_id}')
    sample_records = list(dataset.records(record_set=main_record_set_id))
    print(f'Number of records in this record set: {len(sample_records)}')
    for rec in sample_records[:3]:  # Show 3 example records
        print(rec)


## 3. Data Extraction

Load each available record set into a Pandas DataFrame using only their `@id` as keys. All field columns will also be referenced by their entity `@id`s.

In [ ]:
# Extract all record sets into DataFrames by their @id
all_dataframes = {}
for rs in record_sets:
    rs_id = rs.id
    df = pd.DataFrame(list(dataset.records(record_set=rs_id)))
    all_dataframes[rs_id] = df
    print(f'Record set {rs_id} loaded: {df.shape[0]} rows, {df.shape[1]} columns')
    print(f'Column @id list for {rs_id}:')
    print(list(df.columns))

# Select primary record set for further analysis
if not record_sets:
    main_df = pd.DataFrame()
    print('No DataFrame available for analysis.')
else:
    main_record_set_id = record_sets[0].id
    main_df = all_dataframes[main_record_set_id]
    print(f'Showing head of DataFrame for record set @id={main_record_set_id}:')
    display(main_df.head())

## 4. Exploratory Data Analysis (EDA)

We'll perform some common EDA steps:
- Filtering records on a numeric field (e.g., age, using only its `@id`)
- Normalizing a numeric column by its mean and std (referenced by `@id`)
- Grouping by a categorical field (using `@id`)

**First, identify a numeric and a categorical field by their `@id` for the main table.**

In [ ]:
# Identify a numeric and group (categorical) field by @id from record set fields
if not record_sets or not main_rs.fields:
    print('No fields found for EDA.')
else:
    numeric_field_id = None
    group_field_id = None
    for field in main_rs.fields:
        if getattr(field, 'data_type', '').lower() in {'integer', 'float', 'number'} and numeric_field_id is None:
            numeric_field_id = field.id
        elif getattr(field, 'data_type', '').lower() in {'string', 'text'} and group_field_id is None:
            group_field_id = field.id
        if numeric_field_id and group_field_id:
            break
    print(f'Using numeric field @id: {numeric_field_id}')
    print(f'Using group-by field @id: {group_field_id}')

# --- EDA: filter, normalize, group ---
if not numeric_field_id or numeric_field_id not in main_df.columns:
    print('Could not find numeric field for filtering/normalization.')
else:
    # Convert if needs be
    main_df[numeric_field_id] = pd.to_numeric(main_df[numeric_field_id], errors='coerce')
    threshold = main_df[numeric_field_id].mean()  # For demonstration
    filtered_df = main_df[main_df[numeric_field_id] > threshold].copy()
    print(f"Filtered rows where {numeric_field_id} > {threshold:.2f}: {filtered_df.shape[0]} records")
    print(filtered_df.head())

    # Normalize numeric column
    norm_col = f"{numeric_field_id}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f'Normalized {numeric_field_id} for filtered rows:')
    print(filtered_df[[numeric_field_id, norm_col]].head())

    # Group by a categorical field if available
    if group_field_id and group_field_id in filtered_df.columns:
        grouped = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f'Grouped mean of {numeric_field_id} by {group_field_id}:')
        print(grouped.head())

## 5. Visualization

Plot histograms and grouped bar plots for the selected fields (referenced by their `@id`).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if not numeric_field_id or numeric_field_id not in main_df.columns:
    print('Numeric field not found for plotting.')
else:
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    sns.histplot(main_df[numeric_field_id].dropna(), kde=True, ax=ax[0])
    ax[0].set_title(f'Histogram of {numeric_field_id}')
    ax[0].set_xlabel(numeric_field_id)

    if group_field_id and group_field_id in main_df.columns:
        # Visualize group means
        grouped = main_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        sns.barplot(data=grouped, x=group_field_id, y=numeric_field_id, ax=ax[1])
        ax[1].set_title(f'Mean {numeric_field_id} by {group_field_id}')
        ax[1].set_xticklabels(ax[1].get_xticklabels(), rotation=45, ha='right')
    else:
        ax[1].set_visible(False)
    plt.tight_layout()
    plt.show()


## 6. Conclusion

In this notebook, we demonstrated how to:
- Load and inspect a Croissant-packaged dataset using the `mlcroissant` library
- Explore record sets, fields, and records by their unique `@id`
- Transform, analyze, and visualize data strictly using schema entity identifiers

Such a workflow supports automated, reproducible, and schema-compliant analyses for FAIR research data. Adapt the filtering/grouping parameters above for your own domain insights.